# Sentinel Protocol — Package Analysis Notebook

This notebook imports from the `sentinel` package (refactored from the original monolithic notebook)
and runs exploratory analysis across all threat scenarios.

**Run from the project root** so that `import sentinel` resolves correctly:
```bash
jupyter lab   # from Sentinel Protocol/
```

In [ ]:
import sys, os
# Ensure the project root is on the path when launched from notebooks/
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath('__file__')), '..'))

from sentinel.decision_engine import (
    DecisionTier, Threat, THREAT_CONSERVATISM, classify_threat
)
from sentinel.simulator import TickState, run_scenario, choose_holding_action
from sentinel.safety_gate import (
    SafetyCheckResult, ValidationResult, SurvivalStep,
    is_action_safe, validate_command, blackout_survival_loop,
)
from sentinel.reasoning import generate_reasoning, make_block_report

print('sentinel package imported successfully.')

## 1. classify_threat() — all five threat types

In [ ]:
COMM_DELAY = 780  # s  (~13 min, typical Mars opposition)

test_cases = [
    # (threat_type,          time_to_harm_s, expected_tier)
    ("cliff_edge",       5000,  "GREEN" ),   # adj=4000 > 2*RTT(3120)
    ("cliff_edge",       2500,  "YELLOW"),   # adj=2000, RTT=1560 → 1<2000/1560<2
    ("cliff_edge",        500,  "RED"   ),   # adj=400  < RTT
    ("dust_storm",       4000,  "GREEN" ),   # adj=3600 > 2*RTT
    ("dust_storm",       1800,  "YELLOW"),   # adj=1620, RTT=1560 → ratio=1.038
    ("dust_storm",        400,  "RED"   ),
    ("battery_critical", 4000,  "GREEN" ),
    ("battery_critical", 1800,  "YELLOW"),   # adj=1710, RTT=1560 → ratio=1.096
    ("battery_critical",  200,  "RED"   ),
    ("rockfall",         5000,  "GREEN" ),   # adj=3500 > 2*RTT
    ("rockfall",         3000,  "YELLOW"),   # adj=2100, RTT=1560 → ratio=1.346
    ("rockfall",          300,  "RED"   ),
    ("comms_blackout",   4000,  "GREEN" ),
    ("comms_blackout",   1800,  "YELLOW"),
    ("comms_blackout",    500,  "RED"   ),
]

print(f"{'Threat':<20} {'TTH(s)':>8} {'Expected':>10} {'Got':>10} {'Pass':>6}")
print('-' * 60)
for threat_type, tth, expected in test_cases:
    result = classify_threat(threat_type, tth, COMM_DELAY)
    ok = '✅' if result.value == expected else '❌'
    print(f"{threat_type:<20} {tth:>8} {expected:>10} {result.value:>10} {ok:>6}")

## 2. run_scenario() — full tick trace for each threat type

In [ ]:
import pandas as pd

THREAT_TYPES = ["cliff_edge", "dust_storm", "battery_critical", "rockfall", "comms_blackout"]
TICKS = 15

for threat in THREAT_TYPES:
    rows = []
    for ts in run_scenario(threat, ticks=TICKS, comm_delay_s=COMM_DELAY):
        rows.append({
            "tick":           ts.tick,
            "tier":           ts.tier.value,
            "time_to_harm_s": ts.time_to_harm_s,
            "holding_action": ts.holding_action or "",
            **ts.sensors,
        })
    df = pd.DataFrame(rows)
    print(f"\n{'─'*60}")
    print(f"  Scenario: {threat}")
    print(f"{'─'*60}")
    print(df.to_string(index=False))

## 3. choose_holding_action() — YELLOW-tier holding action logic

In [ ]:
ha_cases = [
    ("cliff_edge",       {"distance_m": 50.0, "drift_speed_ms": 0.05}),
    ("dust_storm",       {"wind_speed_ms": 8.0}),       # below 20 m/s → reposition
    ("dust_storm",       {"wind_speed_ms": 25.0}),      # above 20 m/s → hold
    ("battery_critical", {"charge_pct": 15.0}),          # above 5 % → reposition
    ("battery_critical", {"charge_pct": 3.0}),           # at/below 5 % → hold
    ("rockfall",         {"debris_dist_m": 40.0, "debris_speed_ms": 3.0}),
    ("comms_blackout",   {"relay_elevation_deg": 15.0}),
]

print(f"{'Threat':<20} {'Sensors':<45} {'Action'}")
print('-' * 80)
for threat, sensors in ha_cases:
    action = choose_holding_action(threat, sensors)
    print(f"{threat:<20} {str(sensors):<45} {action}")

## 4. is_action_safe() — universal pre-execution gate

In [ ]:
gate_cases = [
    # action,                   sensors,                                           threats
    ("move_forward",   {"distance_m": 0.5, "drift_speed_ms": 0.02},               ["cliff_edge"]),
    ("move_forward",   {"distance_m": 500.0, "drift_speed_ms": 0.02},             ["cliff_edge"]),
    ("deploy_antenna", {"wind_speed_ms": 22.0, "optical_depth": 0.3},             ["dust_storm"]),
    ("transmit_data",  {"relay_elevation_deg": 5.0},                              ["comms_blackout"]),
    ("hold_in_place",  {},                                                          ["cliff_edge", "rockfall"]),
    ("run_diagnostics",{"charge_pct": 4.0},                                        ["battery_critical"]),
    ("run_diagnostics",{"charge_pct": 20.0},                                       ["battery_critical"]),
]

print(f"{'Action':<18} {'Safe':>5} {'Blocked by':<20} Reason")
print('-' * 80)
for action, sensors, threats in gate_cases:
    r = is_action_safe(action, sensors, threats, comm_delay_s=780)
    print(f"{action:<18} {'✅' if r.safe else '❌':>5} {r.blocked_by:<20} {r.reason}")

## 5. validate_command() — Earth command validator

In [ ]:
vc_cases = [
    # command,           sensor_state,                                  threat_type
    ("move_forward",   {"distance_m": 0.5, "drift_speed_ms": 0.02},    "cliff_edge"),
    ("move_forward",   {"distance_m": 500.0, "drift_speed_ms": 0.02},  "cliff_edge"),
    ("deploy_antenna", {"wind_speed_ms": 22.0, "optical_depth": 0.3},  "dust_storm"),
    ("transmit_data",  {"relay_elevation_deg": 5.0},                   "comms_blackout"),
    ("stop",           {"charge_pct": 4.0},                             "battery_critical"),
]

print(f"{'Command':<18} {'Threat':<20} {'Verdict':<10} Reason")
print('-' * 85)
for cmd, sensors, threat in vc_cases:
    vr = validate_command(cmd, sensors, threat, comm_delay_s=780)
    print(f"{cmd:<18} {threat:<20} {vr.verdict:<10} {vr.reason}")

## 6. blackout_survival_loop() — autonomous comms-blackout survival

In [ ]:
# Scenario: rover near a cliff with moderate charge during comms blackout
initial_sensors = {
    "relay_elevation_deg": 6.0,
    "charge_pct": 20.0,
    "distance_m": 50.0,
    "drift_speed_ms": 0.0,
}

print("Blackout Survival Loop — step-by-step:")
print('=' * 70)
for step in blackout_survival_loop(initial_sensors, comm_delay_s=780, max_wait_steps=4):
    status = '✅ executed' if step.executed else '❌ blocked'
    print(f"  [{step.phase:>14}] {step.proposed:<30} {status}")
    print(f"               ↳ {step.note}")
    print()

## 7. AI reasoning via watsonx.ai (optional)

Requires valid credentials in `.env` at the project root.

In [ ]:
# Sample tick data for a RED-tier rockfall event
tick_data = {
    "threat_type":    "rockfall",
    "sensors":        "{'seismic_g': 0.37, 'debris_dist_m': 12.0, 'debris_speed_ms': 9.5}",
    "time_to_harm_s": 1.3,
    "round_trip_s":   1560.0,
    "ratio":          0.001,
    "tier":           "RED",
    "action":         "Act autonomously NOW; notify Earth after action.",
}

log_entry = generate_reasoning(tick_data)
print("Mission log entry:")
print(log_entry)